# 5G NR PUSCH Tutorial

This notebook provides an introduction to Sionna's [5G New Radio (NR) module](https://nvlabs.github.io/sionna/api/nr.html) and, in particular, the [physical uplink shared channel (PUSCH)](https://nvlabs.github.io/sionna/api/nr.html#pusch). This module provides implementations of a small subset of the physical layer functionalities as described in the 3GPP specifications [38.211](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3213), [38.212](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3214) and [38.214](https://portal.3gpp.org/desktopmodules/Specifications/SpecificationDetails.aspx?specificationId=3216). 


You will

- Get an understanding of the different components of a PUSCH configuration, such as the carrier, DMRS, and transport block,
- Learn how to rapidly simulate PUSCH transmissions for multiple transmitters,
- Modify the PUSCHReceiver to use a custom MIMO Detector.

## Table of Contents
* [GPU Configuration and Imports](#GPU-Configuration-and-Imports)
* [A "Hello World!" Example](#A-Hello-World-Example)
* [Carrier Configuration](#Carrier-Configuration)
* [Understanding the DMRS Configuration](#Understanding-the-DMRS-Configuration)
    * [Configuring Multiple Layers](#Configuring-Multiple-Layers)
    * [Controlling the Number of DMRS Symbols in a Slot](#Controlling-the-Number-of-DMRS-Symbols-in-a-Slot)
    * [How to control the number of available DMRS ports?](#How-to-control-the-number-of-available-DMRS-ports?)
* [Transport Blocks and MCS](#Transport-Blocks-and-MCS)
* [Looking into the PUSCHTransmitter](#Looking-into-the-PUSCHTransmitter)
* [Components of the PUSCHReceiver](#Components-of-the-PUSCHReceiver)
* [End-to-end PUSCH Simulations](#End-to-end-PUSCH-Simulations)


## GPU Configuration and Imports

In [2]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tahtan
# Import Sionna
import sys
sys.path.append('../')
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import * 
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper


In [3]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm.notebook import tqdm
import itertools
import io

## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [4]:
df = pd.read_parquet('/workspaces/thanh/Dataset/tmp/parquet/20250217.parquet')
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2400 entries, 0 to 1199
Data columns (total 59 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   nSFN                      2400 non-null   int64  
 1   nSlot                     2400 non-null   int64  
 2   nPDU                      2400 non-null   int64  
 3   nGroup                    2400 non-null   int64  
 4   nUlsch                    2400 non-null   int64  
 5   nUlcch                    2400 non-null   int64  
 6   nRachPresent              2400 non-null   int64  
 7   nRNTI                     2400 non-null   int64  
 8   nUEId                     2400 non-null   int64  
 9   nBWPSize                  2400 non-null   int64  
 10  nBWPStart                 2400 non-null   int64  
 11  nSubcSpacing              2400 non-null   int64  
 12  nCpType                   2400 non-null   int64  
 13  nULType                   2400 non-null   int64  
 14  nMcsTable    

Hopefully you have enjoyed this tutorial on Sionna's 5G NR PUSCH module!

Please have a look at the [API documentation](https://nvlabs.github.io/sionna/api/sionna.html) of the various components or the other available [tutorials](https://nvlabs.github.io/sionna/tutorials.html) to learn more.

In [5]:
df['nSFN'].unique()

array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [6]:
df['nSlot'].unique()

array([ 4,  5, 14, 15])

In [7]:
slot4cfg = df[df['nSlot'] == 4].iloc[0]

In [8]:
slot5cfg = df[df['nSlot'] == 5].iloc[0]

In [9]:
slot14cfg = df[df['nSlot'] == 14].iloc[0]

In [10]:
slot15cfg = df[df['nSlot'] == 15].iloc[0]

In [11]:
class MyPUSCHConfig(PUSCHConfig):
    def __init__(self):
        super().__init__(
            carrier_config=CarrierConfig(
                n_cell_id=0,
                cyclic_prefix="normal",
                subcarrier_spacing=30,
                n_size_grid=273,
                n_start_grid=0,
                slot_number=4,
                frame_number=0
            ),
            pusch_dmrs_config=PUSCHDMRSConfig(
                config_type=1,
                length=1,
                additional_position=1,
                dmrs_port_set=[0],
                n_id=0,
                n_scid=0,
                num_cdm_groups_without_data=2,
                type_a_position=2
            ),
            tb_config=TBConfig(
                channel_type='PUSCH',
                n_id=0,
                mcs_table=1,
                mcs_index=9
            ),
            mapping_type='A',
            n_size_bwp=273,
            n_start_bwp=0,
            num_layers=1,
            num_antenna_ports=1,
            precoding='non-codebook',
            tpmi=0,
            transform_precoding=False,
            n_rnti=2008,
            symbol_allocation=[0,14]
        )

_pusch_config_0 = MyPUSCHConfig()

_tb_size = _pusch_config_0.tb_size
_num_coded_bits = _pusch_config_0.num_coded_bits
_target_coderate = _pusch_config_0.tb.target_coderate
_num_bits_per_symbol = _pusch_config_0.tb.num_bits_per_symbol
_mu = _pusch_config_0.carrier.mu

_num_layers = _pusch_config_0.num_layers
_n_rnti = _pusch_config_0.n_rnti
_n_id = _pusch_config_0.tb.n_id

_dmrs_length = _pusch_config_0.dmrs.length
_dmrs_additional_position = _pusch_config_0.dmrs.additional_position
_num_cdm_groups_without_data = _pusch_config_0.dmrs.num_cdm_groups_without_data
_n_scid = _pusch_config_0.dmrs.n_scid
_n_id_n_scid = _pusch_config_0.dmrs.n_id[0]

In [12]:
_num_tx = 1
_num_rx = 1
_num_tx_ant = 1
_num_rx_ant = 8
_carrier_frequency = 2.55e9  # Carrier frequency in Hz.
_link_direction = "uplink"

_bandwidth = 100
_nfft = 4096

In [44]:
slot4cfg.nSFN

0

In [31]:
def setCfgReq(dir):
    filename = os.path.join(dir, 'cfgReq.cfg')

    with open(filename, 'w') as file:
        file.write("""\
<?xml version="1.0"?>

<TestConfig>
	<numSlots>20</numSlots>
	<uliq_car0_ant0>rx_ant_0.bin</uliq_car0_ant0>
	<uliq_car0_ant1>rx_ant_1.bin</uliq_car0_ant1>
	<uliq_car0_ant2>rx_ant_2.bin</uliq_car0_ant2>
	<uliq_car0_ant3>rx_ant_3.bin</uliq_car0_ant3>
	<uliq_car0_ant4>rx_ant_4.bin</uliq_car0_ant4>
	<uliq_car0_ant5>rx_ant_5.bin</uliq_car0_ant5>
	<uliq_car0_ant6>rx_ant_6.bin</uliq_car0_ant6>
	<uliq_car0_ant7>rx_ant_7.bin</uliq_car0_ant7>
	<ul_ref_out>ref.txt</ul_ref_out>
	<start_frame_number>0</start_frame_number>
	<start_slot_number>0</start_slot_number>
</TestConfig>

<ConfigReq>
	<nCarrierIdx>0</nCarrierIdx>
	<nDMRSTypeAPos>2</nDMRSTypeAPos>
	<nPhyCellId>0</nPhyCellId>
	<nDLBandwidth>100</nDLBandwidth>
	<nULBandwidth>100</nULBandwidth>
	<nDLFftSize>4096</nDLFftSize>
	<nULFftSize>4096</nULFftSize>
	<nNrOfTxAnt>8</nNrOfTxAnt>
	<nNrOfRxAnt>8</nNrOfRxAnt>
	<nCarrierAggregationLevel>0</nCarrierAggregationLevel>
	<nFrameDuplexType>1</nFrameDuplexType>
	<nSubcCommon>1</nSubcCommon>
	<nTddPeriod>20</nTddPeriod>
	<sSlotConfig0>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig0>
	<sSlotConfig1>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig1>
	<sSlotConfig2>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig2>
	<sSlotConfig3>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig3>
	<sSlotConfig4>1,1,1,1,1,1,1,1,1,1,1,1,1,1</sSlotConfig4>
	<sSlotConfig5>1,1,1,1,1,1,1,1,1,1,1,1,1,1</sSlotConfig5>
	<sSlotConfig6>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig6>
	<sSlotConfig7>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig7>
	<sSlotConfig8>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig8>
	<sSlotConfig9>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig9>
	<sSlotConfig10>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig10>
	<sSlotConfig11>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig11>
	<sSlotConfig12>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig12>
	<sSlotConfig13>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig13>
	<sSlotConfig14>1,1,1,1,1,1,1,1,1,1,1,1,1,1</sSlotConfig14>
	<sSlotConfig15>1,1,1,1,1,1,1,1,1,1,1,1,1,1</sSlotConfig15>
	<sSlotConfig16>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig16>
	<sSlotConfig17>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig17>
	<sSlotConfig18>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig18>
	<sSlotConfig19>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig19>
	<nCyclicPrefix>0</nCyclicPrefix>
</ConfigReq>

<RxConfig>
	<SlotNum0>null.cfg</SlotNum0>
	<SlotNum1>null.cfg</SlotNum1>
	<SlotNum2>null.cfg</SlotNum2>
	<SlotNum3>null.cfg</SlotNum3>
	<SlotNum4>slot4.cfg</SlotNum4>
	<SlotNum5>slot5.cfg</SlotNum5>
	<SlotNum6>null.cfg</SlotNum6>
	<SlotNum7>null.cfg</SlotNum7>
	<SlotNum8>null.cfg</SlotNum8>
	<SlotNum9>null.cfg</SlotNum9>
	<SlotNum10>null.cfg</SlotNum10>
	<SlotNum11>null.cfg</SlotNum11>
	<SlotNum12>null.cfg</SlotNum12>
	<SlotNum13>null.cfg</SlotNum13>
	<SlotNum14>slot14.cfg</SlotNum14>
	<SlotNum15>slot15.cfg</SlotNum15>
	<SlotNum16>null.cfg</SlotNum16>
	<SlotNum17>null.cfg</SlotNum17>
	<SlotNum18>null.cfg</SlotNum18>
	<SlotNum19>null.cfg</SlotNum19>
</RxConfig>
""")

setCfgReq('../ATest')

In [39]:
slot_number = _pusch_config_0.carrier.slot_number
rnti = _pusch_config_0.n_rnti
mcs = _pusch_config_0.tb.mcs_index
tbsize = _pusch_config_0.tb_size

In [ ]:
def setUlCfgReq(slotcfg, dir):
    filename = os.path.join(dir, f'slot{slotcfg.nSlot}.cfg')
	
    with open(filename, 'w') as file:
        file.write(
f"""<?xml version="1.0"?>

<Ul_Config_Req>

	<UlConfigReqL1L2Header>
		<nSFN>{slotcfg.nSFN}</nSFN>
		<nSlot>{slotcfg.nSlot}</nSlot>
		<nPDU>1</nPDU>
		<nGroup>1</nGroup>
		<nUlsch>1</nUlsch>
		<nUlcch>0</nUlcch>
		<nRachPresent>0</nRachPresent>
	</UlConfigReqL1L2Header>

	<UL_SCH_PDU0>
		<nRNTI>{slotcfg.nSlot}</nRNTI>
		<nUEId>0</nUEId>
		<nBWPSize>{slotcfg.nBWPSize}</nBWPSize>
		<nBWPStart>{slotcfg.nBWPStart}</nBWPStart>
		<nSubcSpacing>1</nSubcSpacing>
		<nCpType>0</nCpType>
		<nULType>0</nULType>
		<nMcsTable>{slotcfg}</nMcsTable>
		<nMCS>{slotcfg}</nMCS>
		<nTransPrecode>0</nTransPrecode>
		<nTransmissionScheme>0</nTransmissionScheme>
		<nNrOfLayers>1</nNrOfLayers>
		<nPortIndex0>0</nPortIndex0>
		<nNid>{slotcfg}</nNid>
		<nSCID>{slotcfg}</nSCID>
		<nNIDnSCID>{slotcfg}</nNIDnSCID>
		<nNrOfAntennaPorts>8</nNrOfAntennaPorts>
		<nVRBtoPRB>0</nVRBtoPRB>
		<nPMI>0</nPMI>
		<nStartSymbolIndex>0</nStartSymbolIndex>
		<nNrOfSymbols>14</nNrOfSymbols>
		<nResourceAllocType>1</nResourceAllocType>
		<nRBStart>0</nRBStart>
		<nRBSize>273</nRBSize>
		<nTBSize>{slotcfg}</nTBSize>
		<nRV>0</nRV>
		<nHARQID>{harq_id}</nHARQID>
		<nNDI>1</nNDI>
		<nMappingType>0</nMappingType>
		<nDMRSConfigType>0</nDMRSConfigType>
		<nNrOfCDMs>2</nNrOfCDMs>
		<nNrOfDMRSSymbols>1</nNrOfDMRSSymbols>
		<nDMRSAddPos>1</nDMRSAddPos>
		<nPTRSPresent>0</nPTRSPresent>
		<nAck>0</nAck>
		<nAlphaScaling>0</nAlphaScaling>
		<nBetaOffsetACKIndex>0</nBetaOffsetACKIndex>
		<nCsiPart1>0</nCsiPart1>
		<nBetaOffsetCsiPart1Index>0</nBetaOffsetCsiPart1Index>
		<nCsiPart2>0</nCsiPart2>
		<nBetaOffsetCsiPart2Index>0</nBetaOffsetCsiPart2Index>
		<nTpPi2BPSK>0</nTpPi2BPSK>
		<nTPPuschID>0</nTPPuschID>
		<nRxRUIdx0>0</nRxRUIdx0>
		<nRxRUIdx1>1</nRxRUIdx1>
		<nRxRUIdx2>2</nRxRUIdx2>
		<nRxRUIdx3>3</nRxRUIdx3>
		<nRxRUIdx4>4</nRxRUIdx4>
		<nRxRUIdx5>5</nRxRUIdx5>
		<nRxRUIdx6>6</nRxRUIdx6>
		<nRxRUIdx7>7</nRxRUIdx7>
	</UL_SCH_PDU0>

	<PUSCH_GROUP_INFO0>
		<nUE>1</nUE>
		<nPduIdx0>0</nPduIdx0>
	</PUSCH_GROUP_INFO0>

</Ul_Config_Req>
""")
setUlCfgReq(slot4cfg, '../ATest')

NameError: name 'mcs_table' is not defined

In [20]:
def setCfgReg(pusch_config, dir):
    filename = os.path.join(dir, 'cfgReq.cfg')

    with open(filename, 'w') as file:
        file.write('<?xml version="1.0"?>\n\n')
        file.write('<TestConfig>\n')
        
        file.write(f'\t<numSlots>{pusch_config.carrier.num_slots_per_frame}</numSlots>\n')

        for rxIdx in range(_num_rx_ant):
            file.write(f'\t<uliq_car0_ant{rxIdx}>rx_ant_{rxIdx}.bin</uliq_car0_ant{rxIdx}>\n')

        file.write('\t<ul_ref_out>ref.txt</ul_ref_out>\n')
        FrameNumber = 0
        file.write(f'\t<start_frame_number>{FrameNumber}</start_frame_number>\n')
        file.write(f'\t<start_slot_number>{0}</start_slot_number>\n')
        file.write('</TestConfig>\n\n')

        file.write('<ConfigReq>\n')
        file.write('\t<nCarrierIdx>0</nCarrierIdx>\n')
        file.write(f'\t<nDMRSTypeAPos>{pusch_config.dmrs.type_a_position}</nDMRSTypeAPos>\n')
        file.write(f'\t<nPhyCellId>{pusch_config.carrier.n_cell_id}</nPhyCellId>\n')

        file.write(f'\t<nDLBandwidth>{_bandwidth}</nDLBandwidth>\n')  # Hardcoded same as UL
        file.write(f'\t<nULBandwidth>{_bandwidth}</nULBandwidth>\n')  # Hardcoded same as UL

        # Nfft = 2**np.ceil(np.log2(pusch_config.carrier.n_size_grid * 12 / 0.85))
        # Nfft = max(128, int(Nfft)) 
        file.write(f'\t<nDLFftSize>{_nfft}</nDLFftSize>\n')
        file.write(f'\t<nULFftSize>{_nfft}</nULFftSize>\n')

        file.write(f'\t<nNrOfTxAnt>{_num_rx_ant}</nNrOfTxAnt>\n')
        file.write(f'\t<nNrOfRxAnt>{_num_rx_ant}</nNrOfRxAnt>\n')
        file.write(f'\t<nCarrierAggregationLevel>0</nCarrierAggregationLevel>\n')
        file.write(f'\t<nFrameDuplexType>1</nFrameDuplexType>\n')
        file.write(f'\t<nSubcCommon>{_mu}</nSubcCommon>\n')
        file.write(f'\t<nTddPeriod>20</nTddPeriod>\n')

        slotSetInFrame = [4,5,14,15]
        for slotIdx in range(pusch_config.carrier.num_slots_per_frame):
            if slotIdx in slotSetInFrame:
                file.write(f'\t<sSlotConfig{slotIdx}>1,1,1,1,1,1,1,1,1,1,1,1,1,1</sSlotConfig{slotIdx}>\n')
            else:
                file.write(f'\t<sSlotConfig{slotIdx}>2,2,2,2,2,2,2,2,2,2,2,2,2,2</sSlotConfig{slotIdx}>\n')

        cpType = int(pusch_config.carrier.cyclic_prefix!='normal')
        file.write(f'\t<nCyclicPrefix>{cpType}</nCyclicPrefix>\n')
        file.write('</ConfigReq>\n\n')

        file.write('<RxConfig>\n')
        for slotIdx in range(pusch_config.carrier.num_slots_per_frame):
            if slotIdx in slotSetInFrame:
                file.write(f'\t<SlotNum{slotIdx}>slot{slotIdx}.cfg</SlotNum{slotIdx}>\n')
            else:
                file.write(f'\t<SlotNum{slotIdx}>null.cfg</SlotNum{slotIdx}>\n')

        file.write('</RxConfig>\n')

setCfgReg(_pusch_config_0, '../ABC')

In [34]:
with open('/workspaces/thanh/ABC/slot4.cfg', 'r') as f:
    ccc = f.read()
with open('/workspaces/thanh/ATest/slot4.cfg', 'r') as f:
    bbb = f.read()

In [35]:
len(bbb),len(ccc)

(1885, 1884)

In [27]:
bbb[90:94], bbb[80:100]

('>6</', '>\n\t\t<nSlot>6</nSlot>')

In [26]:
i = 0
b,c = 0,0
while(b == c) :
    b = bbb[i]
    c = ccc[i]
    i+=1
print(i,b,c)

92 6 4


In [ ]:
for n,(b,c) in enumerate(list(zip(bbb,ccc))):
    

< <
? ?
x x
m m
l l
   
v v
e e
r r
s s
i i
o o
n n
= =
" "
1 1
. .
0 0
" "
? ?
> >

 


 

< <
T T
e e
s s
t t
C C
o o
n n
f f
i i
g g
> >

 

	 	
< <
n n
u u
m m
S S
l l
o o
t t
s s
> >
2 2
0 0
< <
/ /
n n
u u
m m
S S
l l
o o
t t
s s
> >

 

	 	
< <
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
0 0
> >
r r
x x
_ _
a a
n n
t t
_ _
0 0
. .
b b
i i
n n
< <
/ /
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
0 0
> >

 

	 	
< <
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
1 1
> >
r r
x x
_ _
a a
n n
t t
_ _
1 1
. .
b b
i i
n n
< <
/ /
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
1 1
> >

 

	 	
< <
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
2 2
> >
r r
x x
_ _
a a
n n
t t
_ _
2 2
. .
b b
i i
n n
< <
/ /
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
2 2
> >

 

	 	
< <
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
3 3
> >
r r
x x
_ _
a a
n n
t t
_ _
3 3
. .
b b
i i
n n
< <
/ /
u u
l l
i i
q q
_ _
c c
a a
r r
0 0
_ _
a a
n n
t t
3 3
> >

 

	 	
